# Bitcoin GRU Price Direction Prediction

Cleaned notebook for predicting the next Bitcoin price direction with a 7-day sliding window and a stacked GRU classifier. Colab metadata, local paths, and execution outputs were removed for publication.

# 프리미엄과 금 데이터 제외 데이터(0.8)

## 데이터 전처리

In [ ]:
# 라이브러리 임포트
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings(action='ignore')

import matplotlib.pyplot as plt

# 성능지표
from sklearn.metrics import confusion_matrix, classification_report
import sklearn.metrics as mt

from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score 
from sklearn.metrics import precision_score
from sklearn.metrics import f1_score

In [ ]:
data = pd.read_csv('../data/real_final_data.csv', encoding='cp949')

In [ ]:
data.info()

In [ ]:
data.head()

선형 보간법

In [ ]:
# 선형 보간법으로 결측치를 없앰
data1 = data.interpolate(method="linear")

In [ ]:
data1.head(3)

데이터 셋 분류

In [ ]:
# 훈련데이터 셋과 테스트데이터 셋을 분류
train = data1.iloc[:-37,:]
test = data1.iloc[-37:,:]

In [ ]:
# 각 데이터 셋에서 Feature와 label을 분류
train_feature = train.iloc[:,2:]
train_label = train.iloc[:,1]

test_feature = test.iloc[:,2:]
test_label = test.iloc[:,1]

원핫인코딩 처리

In [ ]:
# label은 원핫인코딩 처리
train_label1 = pd.get_dummies(train_label)
test_label1 = pd.get_dummies(test_label)

In [ ]:
test_label1.head(5)

데이터 표준화를 사용

In [ ]:
# 데이터 표준화 (StandardScaler)
from sklearn.preprocessing import StandardScaler

# 훈련 데이터셋 feature
scaler = StandardScaler()

train_feature = scaler.fit_transform(train_feature)
train_feature = pd.DataFrame(train_feature)

# 테스트 데이터셋 feature
scaler = StandardScaler()

test_feature = scaler.fit_transform(test_feature)
test_feature1 = pd.DataFrame(test_feature)

train_feature.head(3), test_feature1.head(3)

슬라이딩 윈도우 1일로 데이터를 변환

In [ ]:
# (배치 사이즈, 날짜, 속성)으로 데이터 셋을 생성
def make_dataset(data, label, window_size):
    feature_list = []
    label_list = []
    for i in range(len(data) - window_size):
        feature_list.append(np.array(data.iloc[i:i+window_size]))
        label_list.append(np.array(label.iloc[i+window_size]))
    return np.array(feature_list), np.array(label_list)

In [ ]:
# 데이터 셋을 슬라이딩 윈도우 형식으로 만들기
# 7일으로 설정

# 훈련데이터 셋
train_feature1, train_label = make_dataset(train_feature, train_label1, 7)

#테스트 데이터 셋
test_feature, test_label = make_dataset(test_feature1, test_label1, 7)

x_train = train_feature1
y_train = train_label

x_train.shape, y_train.shape, test_label.shape


## 모델링: GRU

> 모델 설명

* 1개의 인풋 레이어와 5개의 히든 레이어, 1개의 아웃풋 레이어로 구성
* 5층의 히든 레이어는 각각 32, 64, 128, 256, 512개의 노드로 구성됨
* 각 노드는 GRU를 사용하였으며, 활성화 함수로 Tanh를 사용함
* 아웃풋 레이어는 2개의 노드이며, 활성화 함수로 softmax를 사용함

> 모델 학습 설명
* 모델 학습의 손실함수로는 binary_crossentropy를 사용함
* 최적화로는 adam을 사용함
* epochs는 31번으로 구성됨

In [ ]:
# 모델 생성
from keras.models import Sequential
from keras.layers import Dense
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.layers import LSTM, GRU
import tensorflow as tf

tf.random.set_seed(2)

model = Sequential()

model.add(GRU(32, 
               input_shape=(train_feature1.shape[1], train_feature1.shape[2]), 
               activation='tanh', return_sequences = True))

model.add(GRU(64, 
               activation='tanh', return_sequences = True))

model.add(GRU(128, 
               activation='tanh', return_sequences = True))

model.add(GRU(256, 
               activation='tanh', return_sequences = True))

model.add(GRU(512, 
               activation='tanh'))

model.add(Dense(2, activation='softmax'))

In [ ]:
# 모델 확인
model.summary()

In [ ]:
# 모델 학습
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

history = model.fit(x_train, y_train, batch_size=31, epochs=31)  

In [ ]:
# 모델 loss 그래프

y_test_loss = history.history['accuracy']
y_train_loss = history.history['loss']

x_len = np.arange(len(y_test_loss))
plt.plot(x_len, y_test_loss, marker=',', c='red', label='Accuracy')
plt.plot(x_len, y_train_loss, marker=',', c='blue', label='Trainset_loss')

plt.legend(loc='upper right')
plt.grid()
plt.xlabel('epoch')
plt.ylabel('loss')
plt.show()

모델 예측

In [ ]:
# 모델 예측
pred = model.predict(test_feature)
pred

예측값을 원핫인코딩으로 전환

In [ ]:
# 위치를 숫자로 반환
preds = np.argmax(pred, axis=1)
preds

In [ ]:
# 원핫인코딩
preds1 = pd.get_dummies(preds)
preds1

In [ ]:
plt.figure(figsize=(12, 9))
plt.scatter(pd.DataFrame(test_label).index, test_label[:,1], label = 'actual', s = 400, c = 'gray')
plt.scatter(pd.DataFrame(test_label).index, preds1[1], label = 'prediction', s= 50, c = 'red')
plt.ylim([-0.5, 1.5])
plt.title("Upward") 
plt.legend()
plt.show()

성능지표

In [ ]:
test_label1 = np.array(test_label).argmax(axis=1)

# 성능평가: 하나로 묶어서
print('accuracy',mt.accuracy_score(test_label1,preds))
print(confusion_matrix(test_label1, preds))
print(classification_report(test_label1, preds, target_names=['하락', '상승']))

# RF를 통해 뽑은 데이터(0.63)

RF를 통해 뽑은 feature_importances 17개

In [ ]:
start = time.time()

# 랜덤으로 할 파라미터 정의
param_list = {"n_estimators": list(range(20, 200, 20)),
              "max_depth": list(range(4, 21, 4)),
              "max_features": list(range(5, 40, 5)),
              "min_samples_split": list(range(3, 13, 2))}

# 하이퍼파라미터 최적화
RF = RandomForestClassifier()
RF_random_search = RandomizedSearchCV(estimator = RF,
                                        param_distributions = param_list,
                                        n_iter = 3,       # 5번 반복하는 랜덤포레스트를 구현
                                        cv = 3,           # 3번의 cross-validation
                                        n_jobs = 10,
                                        random_state=42) 
RF_random_search.fit(X_train, y_train)
y_pred = RF_random_search.predict(X_test)

#성능평가
print('accuracy',mt.accuracy_score(y_test,y_pred))

print( clock(start) )
print( RF_random_search.best_params_ ) #파라미터 중 가장 정확도가 높은 파라미터를 출력
#가장 추정이 잘된 변수명들의 정확도를 순서대로 나열함
f_i1 = pd.DataFrame(sorted(zip(RF_random_search.best_estimator_.feature_importances_*100, X_train.columns), reverse=True), columns=['f_i','columns'])
f_i1.head(20)

데이터 확인

In [ ]:
# 17개의 특징들
Features = ['date'	,'Up/Down', '전일대비', '상승분', '증감률', '하락분', '%K', '%B', '%D', '프리미엄%_업비트', 'outflow_top10', 'reserve_bi', 'mpi', '원/달러(저가)', 'reserve_der', 'reserve_x', '받으실 때', 'reserve_usd_der', 'stock_to_flow']

In [ ]:
# 데이터 확인
data.loc[:,Features]

모델링

In [ ]:
# 모델 생성
from keras.models import Sequential
from keras.layers import Dense
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.layers import LSTM, GRU
import tensorflow as tf

tf.random.set_seed(2)

model = Sequential()

model.add(GRU(32, 
               input_shape=(train_feature1.shape[1], train_feature1.shape[2]), 
               activation='tanh', return_sequences = True))

model.add(GRU(64, 
               activation='tanh', return_sequences = True))

model.add(GRU(128, 
               activation='tanh', return_sequences = True))

model.add(GRU(256, 
               activation='tanh', return_sequences = True))

model.add(GRU(512, 
               activation='tanh'))

model.add(Dense(2, activation='softmax'))

In [ ]:
# 학습된 모델 불러오기
from keras.models import load_model
model = load_model('../models/GRU_SelectedVariables_0.63_model3114.h5')

model.summary()

In [ ]:
# 모델 예측
pred = model.predict(test_feature)

preds = np.argmax(pred, axis=1)

preds1 = pd.get_dummies(preds)

test_label1 = np.array(test_label).argmax(axis=1)

# 성능평가
print('accuracy',mt.accuracy_score(test_label1,preds))
print(confusion_matrix(test_label1, preds))
print(classification_report(test_label1, preds, target_names=['하락', '상승']))

# 모든 데이터(0.73)

데이터

In [ ]:
data = pd.read_csv('../data/final_data_0601_3.csv', encoding='cp949')

In [ ]:
data.info()

In [ ]:
data.head()

모델링

In [ ]:
# 모델 생성
from keras.models import Sequential
from keras.layers import Dense
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.layers import LSTM, GRU
import tensorflow as tf

tf.random.set_seed(2)

model = Sequential()

model.add(GRU(32, 
               input_shape=(train_feature1.shape[1], train_feature1.shape[2]), 
               activation='tanh', return_sequences = True))

model.add(GRU(64, 
              input_shape=(train_feature1.shape[1], train_feature1.shape[2]), 
               activation='tanh', return_sequences = True))

model.add(GRU(128, 
               activation='tanh', return_sequences = True))

model.add(GRU(256, 
               activation='tanh', return_sequences = True))

model.add(GRU(512, 
               activation='tanh'))

model.add(Dense(2, activation='softmax'))

In [ ]:
# 학습된 모델 불러오기
from keras.models import load_model
model1 = load_model('../models/GRU_FullVariables_0.73_model3225.h5')

model1.summary()

In [ ]:
# 모델 예측
pred = model1.predict(test_feature)

preds = np.argmax(pred, axis=1)

preds1 = pd.get_dummies(preds)

test_label1 = np.array(test_label).argmax(axis=1)

# 성능평가
print('accuracy',mt.accuracy_score(test_label1,preds))
print(confusion_matrix(test_label1, preds))
print(classification_report(test_label1, preds, target_names=['하락', '상승']))